# Paper 3 target attack family detection recall

Run this notebook in **Google Colab → CPU runtime → Runtime → Run all**. It uses the completed metadata recovery lookup files already in your Google Drive and the original 180 saved prediction files. No raw-data scan, retraining or GitHub clone is needed.

This is a new post hoc supplementary analysis. It retains every target family separately, reports support counts and detection recall, flags groups with fewer than 50 examples, and retains absent families with undefined recall. The 50-example flag is descriptive; it is not a significance cutoff. There are no new p values or confidence intervals.

Before accepting each job it must reproduce the original binary confusion counts, attack/benign recalls and balanced accuracy. Original models, thresholds and saved predictions are unchanged.

This is a fixed binary detector's performance **within target attack families**. It does not perform multiclass classification, hold out attack families from training, harmonise family taxonomies or test family-specific warning prediction. The original panels remain balanced; natural-prevalence evaluation remains separate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
PAPER3_ROOT = Path('/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet')
RECOVERY_ROOT = PAPER3_ROOT / 'supplementary_raw_metadata_recovery_20261024' / 'run_v1'
if not RECOVERY_ROOT.is_dir():
    raise FileNotFoundError(f'Recovered metadata folder not found: {RECOVERY_ROOT}')
print('Using recovered metadata:', RECOVERY_ROOT)

## Calculate and verify all 180 jobs

The lookup/checkpoint hashes below are pinned to the recovery report already supplied for review. Keep the completed recovery files unchanged. A mismatch will stop this analysis and appear in the output report; do not bypass the check.

The code records a plan before calculations, then writes a fresh timestamped results folder. It prints progress every 15 jobs. All exported results are aggregate family counts and verification records; raw lookup files and per-flow predictions are not included.

Each family summary states how many jobs have support and the minimum/maximum sample counts. Means are equal-weight job means. Seed panels overlap and datasets are reused, so neither job counts nor summed evaluations should be treated as independent flows.

In [ ]:
PINNED_RECOVERY = {'NF_BoT_IoT_v3': {'complete_sha256': 'f705142b2ab51d9c1ec549e3ebaba4b5a7018a51bc578acf156d3e2835110602', 'lookup_relative_path': 'attempt_20261024_001733_300411/saved_row_metadata.parquet', 'lookup_sha256': '87cc87a342c865e1221c64e2df36a6396a4bb0606ccc9558a8ca709271377b6c', 'raw_family_relative_path': 'attempt_20261024_001733_300411/raw_family_counts.csv', 'raw_family_sha256': '9a9b1cd5b560f80cc781f4e5d246c9bfb15836118cdc2fb16801557c930cc908'}, 'NF_CSE_CIC_IDS2018_v3': {'complete_sha256': 'ed227b582cd0c2f3280a806e360ecd88aab60c79e8dad1b5666c92a219a53970', 'lookup_relative_path': 'attempt_20261024_001929_957898/saved_row_metadata.parquet', 'lookup_sha256': '414053c4b5dbbbabf5a8913dbe05a050b51fa2da7b46b0d2c469a9fc810b66a4', 'raw_family_relative_path': 'attempt_20261024_001929_957898/raw_family_counts.csv', 'raw_family_sha256': 'c9ff45c486ec6a2cabb8b72d6165370e24093824d87c27f32a33b4ca4ebd1617'}, 'NF_ToN_IoT_v3': {'complete_sha256': '8ad7fa9d74427dba1edc5e3d6164521fe524e0de52b121cdb04d7dad5e831d05', 'lookup_relative_path': 'attempt_20261024_002148_810517/saved_row_metadata.parquet', 'lookup_sha256': '29b65ee2fa145c6a73c34f20640d497f48ef56cf13231d96993ac356d7bf2218', 'raw_family_relative_path': 'attempt_20261024_002148_810517/raw_family_counts.csv', 'raw_family_sha256': '9f9734ba26f5db166a9704e7a86b7f8fce4c3849045bbd0243161077ea8a8f71'}, 'NF_UNSW_NB15_v3': {'complete_sha256': '023f40a0d43ab8c31bdbd6e6527169cdd51dd8015de673af036922a9a9a99bdc', 'lookup_relative_path': 'attempt_20261024_002432_955014/saved_row_metadata.parquet', 'lookup_sha256': '944e9d10b254469ae08c18ef999c79ec3bec5262b713730c15379d7f940580c6', 'raw_family_relative_path': 'attempt_20261024_002432_955014/raw_family_counts.csv', 'raw_family_sha256': 'd1ae4b7b70185e3c78ae5acbbef42d8b0102196117386e04ff12a5ae95eafbc5'}}
EVALUATION_SOURCE = '"""Post hoc family-resolved detection recall from unchanged saved predictions."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib, json, zipfile, platform\nimport numpy as np\nimport pandas as pd\n\nDATASETS=[\'NF_BoT_IoT_v3\',\'NF_CSE_CIC_IDS2018_v3\',\'NF_ToN_IoT_v3\',\'NF_UNSW_NB15_v3\']\nMODELS=[\'LogisticRegression\',\'RandomForest\',\'XGBoost\']\nEXPECTED_METRICS_SHA=\'0f997deebc76fe4ad3ba0fc693622a13abc43573e93e41da05e707438e1595b7\'\n\ndef sha(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(2**20),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef read_json(p):return json.loads(Path(p).read_text())\ndef write_json(p,v):\n    p=Path(p);t=p.with_suffix(\'.tmp\');t.write_text(json.dumps(v,indent=2,allow_nan=False)+\'\\n\');t.replace(p)\ndef require_hash(p,expected):\n    actual=sha(p)\n    if not expected or actual!=expected:raise ValueError(\'File hash differs from verified reference: \'+str(p))\n    return actual\n\ndef family_counts(families,predictions,universe):\n    """Retain absent families as n=0, recall missing; verify pandas against NumPy."""\n    g=pd.DataFrame({\'family\':families,\'detected\':predictions}).groupby(\'family\',sort=True).detected.agg([\'count\',\'sum\'])\n    rows=[]\n    for family in universe:\n        mask=np.asarray(families)==family\n        n=int(mask.sum());tp=int(np.asarray(predictions)[mask].sum())\n        if family in g.index:\n            if int(g.loc[family,\'count\'])!=n or int(g.loc[family,\'sum\'])!=tp:raise ValueError(\'Independent family count disagreement\')\n        elif n or tp:raise ValueError(\'Family grouping lost observations\')\n        rows.append({\'family\':family,\'n_attack\':n,\'detected\':tp,\'missed\':n-tp,\'recall\':tp/n if n else np.nan,\n            \'support_status\':\'NO_SUPPORT\' if n==0 else (\'SMALL_GROUP_LT_50\' if n<50 else \'N_AT_LEAST_50\')})\n    if sum(x[\'n_attack\'] for x in rows)!=len(families):raise ValueError(\'Family universe does not cover all attack rows\')\n    return rows\n\ndef evaluate(root,recovery_root,output):\n    root=Path(root).resolve();recovery_root=Path(recovery_root).resolve();output=Path(output).resolve()\n    if output.parent!=root/\'supplementary_attack_family_evaluation_20261024\':raise ValueError(\'Use the separate family evaluation output folder\')\n    if not root.is_dir() or not recovery_root.is_dir():raise FileNotFoundError(\'Paper 3 or metadata recovery folder missing\')\n    output.mkdir(parents=True,exist_ok=False)\n    plan=\'\'\'Post hoc family-resolved transfer evaluation plan, recorded before computing these family results.\nUse all 180 original target jobs (four datasets, three source classifiers, five seeds, twelve transfer directions); no retraining, calibration, threshold changes or job selection.\nUse the verified raw-metadata recovery lookup and original candidate label vault. Keep exact attack family labels separate by target dataset; do not harmonise names or label attacks seen/unseen across collections.\nFor each job and each raw target attack family report n, detected, missed and recall TP_family/n_family. Retain n=0 with undefined recall, not zero. Retain small families; flag n<50 only as a descriptive display aid, not an inferential threshold.\nReproduce original target confusion counts, binary attack/benign recalls and balanced accuracy before accepting a job. Independently check group counts with NumPy.\nSummaries use equal-weight mean/min/max recall over jobs with support and state numbers of jobs, smallest/largest family sample counts, zero-support and small-support jobs. Sum-of-job observations are repeated evaluation counts, not unique flows; no pooled independent-flow confidence interval is calculated.\nReport every family for every transfer/model/seed, including negative and unsupported results. No p values, superiority tests, family-specific warning calibration or new confirmatory claim.\nThis evaluates a fixed binary detector\'s recall within target attack families. It is not family classification, leave-one-family-out training, open-set detection, natural-prevalence validation or proof that source stability predicts family-specific failure.\n\'\'\'\n    (output/\'PLAN.md\').write_text(plan);(output/\'EVALUATION_CODE.py\').write_text(EVALUATION_SOURCE)\n    write_json(output/\'PINNED_RECOVERY_INPUTS.json\',PINNED_RECOVERY)\n    write_json(output/\'progress.json\',{\'status\':\'RUNNING\',\'completed_jobs\':0})\n    try:\n        metrics_path=root/\'Paper3_Final_Evaluation/target_metrics_all_180.csv\'\n        metrics_sha=require_hash(metrics_path,EXPECTED_METRICS_SHA)\n        original=pd.read_csv(metrics_path)\n        keys=[\'source_dataset\',\'target_dataset\',\'seed\',\'model\']\n        expected={(s,t,seed,m) for s in DATASETS for t in DATASETS if s!=t for seed in range(2027,2032) for m in MODELS}\n        if len(original)!=180 or original.duplicated(keys).any() or set(map(tuple,original[keys].to_numpy()))!=expected:\n            raise ValueError(\'Original target metrics job keys differ from the expected 180 jobs\')\n        orig=original.set_index(keys,verify_integrity=True)\n        families_by_target={};lookups={};vaults={};inputs=[];coverage=[]\n        for dataset in DATASETS:\n            pin=PINNED_RECOVERY[dataset];folder=recovery_root/dataset\n            require_hash(folder/\'complete.json\',pin[\'complete_sha256\'])\n            checkpoint=read_json(folder/\'complete.json\')\n            lookup_path=folder/pin[\'lookup_relative_path\']\n            require_hash(lookup_path,pin[\'lookup_sha256\'])\n            rawpath=folder/pin[\'raw_family_relative_path\'];require_hash(rawpath,pin[\'raw_family_sha256\'])\n            raw=pd.read_csv(rawpath,keep_default_na=False)\n            if not set(raw.label.astype(str)).issubset({\'0\',\'1\'}) or raw.family.astype(str).str.strip().eq(\'\').any():\n                raise ValueError(\'Missing families or invalid raw family labels\')\n            universe=sorted(raw.loc[raw.label.astype(str)==\'1\',\'family\'].astype(str).unique())\n            if not universe:raise ValueError(\'No raw attack family inventory\')\n            families_by_target[dataset]=universe\n            recovered=pd.read_parquet(lookup_path,columns=[\'row_id\',\'Label\',\'Attack\'])\n            if not recovered.row_id.is_unique or not recovered.Label.isin([0,1]).all() or recovered.Attack.isna().any():\n                raise ValueError(\'Recovered lookup IDs/labels/families are invalid\')\n            recovered=recovered.set_index(\'row_id\',verify_integrity=True)\n            vp=root/\'Paper3_Target_Label_Vault\'/dataset/\'target_candidate_labels.parquet\'\n            vkey=str(vp.relative_to(root));require_hash(vp,checkpoint[\'input_hashes\'][vkey])\n            vault=pd.read_parquet(vp,columns=[\'row_id\',\'Label\']).set_index(\'row_id\',verify_integrity=True)\n            if len(vault)!=100000 or vault.Label.value_counts().to_dict()!={0:50000,1:50000}:raise ValueError(\'Original candidate quotas differ\')\n            aligned=recovered.reindex(vault.index)\n            if aligned.Label.isna().any() or not np.array_equal(aligned.Label.to_numpy(),vault.Label.to_numpy()):raise ValueError(\'Candidate labels differ from recovered raw labels\')\n            if not set(aligned.loc[aligned.Label==1,\'Attack\'].astype(str)).issubset(universe):raise ValueError(\'Candidate family outside raw inventory\')\n            lookups[dataset]=recovered;vaults[dataset]=vault\n            for family in universe:\n                coverage.append({\'target_dataset\':dataset,\'family\':family,\n                    \'raw_attack_rows\':int(raw.loc[(raw.label.astype(str)==\'1\')&(raw.family==family),\'rows\'].sum()),\n                    \'candidate_attack_rows\':int(((aligned.Label==1)&(aligned.Attack==family)).sum())})\n            inputs.append({\'dataset\':dataset,\'lookup_sha256\':pin[\'lookup_sha256\'],\'candidate_labels_sha256\':sha(vp)})\n        rows=[];checks=[];prediction_inputs=[]\n        for source,target,seed,model in sorted(expected):\n            key=(source,target,seed,model);old=orig.loc[key]\n            base=root/\'Paper3_Target_Predictions\'/(source+\'__to__\'+target)/f\'seed_{seed}\'\n            p=base/(model+\'_target_predictions.parquet\');digest=require_hash(p,str(old.prediction_file_sha256))\n            mpath=base/(model+\'_prediction_manifest.json\');manifest=read_json(mpath)\n            if manifest[\'prediction_file_sha256\']!=digest:raise ValueError(\'Prediction manifest and original result hash disagree\')\n            f=pd.read_parquet(p,columns=[\'row_id\',\'attack_probability\',\'predicted_label\'])\n            if len(f)!=80000 or not f.row_id.is_unique or not f.predicted_label.isin([0,1]).all():raise ValueError(\'Invalid prediction panel\')\n            pro=f.attack_probability.to_numpy(float);pred=f.predicted_label.to_numpy(int);threshold=float(manifest[\'model_threshold\'])\n            if not (np.isfinite(pro).all() and ((pro>=0)&(pro<=1)).all() and np.isfinite(threshold) and 0<=threshold<=1):raise ValueError(\'Invalid probabilities or threshold\')\n            if not np.array_equal((pro>=threshold).astype(int),pred):raise ValueError(\'Stored predictions disagree with fixed threshold\')\n            labels=vaults[target].Label.reindex(f.row_id)\n            recovered=lookups[target].reindex(f.row_id)\n            if labels.isna().any() or recovered.Label.isna().any() or not np.array_equal(labels.to_numpy(),recovered.Label.to_numpy()):raise ValueError(\'Prediction/label/family join failed\')\n            y=labels.to_numpy(int);attack=y==1;benign=y==0\n            if int(attack.sum())!=40000 or int(benign.sum())!=40000:raise ValueError(\'Target job quotas differ\')\n            confusion=np.bincount(2*y+pred,minlength=4);tn,fp,fn,tp=map(int,confusion)\n            for name,value in zip([\'tn\',\'fp\',\'fn\',\'tp\'],[tn,fp,fn,tp]):\n                if int(old[name])!=value:raise ValueError(\'Original confusion count mismatch: \'+name)\n            br=tn/(tn+fp);ar=tp/(tp+fn);ba=(br+ar)/2\n            for name,value in [(\'target_benign_recall\',br),(\'target_attack_recall\',ar),(\'target_balanced_accuracy\',ba)]:\n                if not np.isclose(value,float(old[name]),rtol=0,atol=1e-12):raise ValueError(\'Original metric mismatch: \'+name)\n            counts=family_counts(recovered.Attack.to_numpy(dtype=str)[attack],pred[attack],families_by_target[target])\n            if sum(x[\'detected\'] for x in counts)!=tp or sum(x[\'missed\'] for x in counts)!=fn:raise ValueError(\'Family decomposition differs from binary outcomes\')\n            for rec in counts:rows.append(dict(zip(keys,key))|rec)\n            checks.append(dict(zip(keys,key))|{\'tn\':tn,\'fp\':fp,\'fn\':fn,\'tp\':tp,\'benign_recall\':br,\'attack_recall\':ar,\'balanced_accuracy\':ba,\n                \'original_counts_and_recalls_match\':True,\'family_decomposition_matches\':True,\'independent_family_counts_match\':True})\n            prediction_inputs.append(dict(zip(keys,key))|{\'prediction_sha256\':digest,\'manifest_sha256\':sha(mpath),\'threshold\':threshold})\n            write_json(output/\'progress.json\',{\'status\':\'RUNNING\',\'completed_jobs\':len(checks),\'expected_jobs\':180})\n            if len(checks)%15==0:print(\'Verified and evaluated:\',len(checks),\'/ 180 jobs\',flush=True)\n        frame=pd.DataFrame(rows)\n        frame.to_csv(output/\'family_recall_per_job.csv\',index=False)\n        pd.DataFrame(checks).to_csv(output/\'binary_reproduction_checks.csv\',index=False)\n        pd.DataFrame(coverage).to_csv(output/\'family_coverage.csv\',index=False)\n        def summarise(group_keys):\n            result=[]\n            for key,g in frame.groupby(group_keys,sort=True):\n                if not isinstance(key,tuple):key=(key,)\n                valid=g[g.n_attack>0]\n                result.append(dict(zip(group_keys,key))|{\'jobs_total\':len(g),\'jobs_with_support\':len(valid),\n                    \'zero_support_jobs\':int((g.n_attack==0).sum()),\'small_support_jobs_lt_50\':int(((g.n_attack>0)&(g.n_attack<50)).sum()),\n                    \'minimum_rows_per_job\':int(g.n_attack.min()),\'maximum_rows_per_job\':int(g.n_attack.max()),\n                    \'mean_recall_over_supported_jobs\':float(valid.recall.mean()) if len(valid) else np.nan,\n                    \'minimum_recall_over_supported_jobs\':float(valid.recall.min()) if len(valid) else np.nan,\n                    \'maximum_recall_over_supported_jobs\':float(valid.recall.max()) if len(valid) else np.nan})\n            return pd.DataFrame(result)\n        summarise([\'source_dataset\',\'target_dataset\',\'model\',\'family\']).to_csv(output/\'family_recall_by_direction_model.csv\',index=False)\n        summarise([\'target_dataset\',\'model\',\'family\']).to_csv(output/\'family_recall_by_target_model.csv\',index=False)\n        write_json(output/\'input_manifest.json\',{\'original_target_metrics_sha256\':metrics_sha,\'recovery_inputs\':inputs,\'prediction_inputs\':prediction_inputs})\n        summary={\'status\':\'PASS\',\'scope\':\'post_hoc_target_family_resolved_binary_detection\',\'jobs_completed\':len(checks),\n            \'family_job_records_including_no_support\':len(frame),\'small_support_family_job_records\':int(((frame.n_attack>0)&(frame.n_attack<50)).sum()),\n            \'zero_support_family_job_records\':int((frame.n_attack==0).sum()),\'original_binary_jobs_reproduced\':len(checks),\n            \'no_retraining\':True,\'no_threshold_changes\':True,\'no_new_p_values_or_confidence_intervals\':True,\n            \'capture_prevalence_not_evaluated\':True,\'family_specific_warning_prediction_not_evaluated\':True,\n            \'source_family_holdout_not_evaluated\':True,\'pandas_version\':pd.__version__,\'numpy_version\':np.__version__,\'python\':platform.python_version(),\n            \'code_sha256\':hashlib.sha256(EVALUATION_SOURCE.encode()).hexdigest(),\'plan_sha256\':sha(output/\'PLAN.md\'),\n            \'interpretation\':\'Descriptive target-family detection recall. Shared datasets, overlapping seed panels and rare families limit interpretation; exact family strings are not harmonised across datasets.\'}\n        write_json(output/\'verification_summary.json\',summary)\n        (output/\'README.md\').write_text(\'\'\'# Post hoc target family recall\nUse family_recall_per_job.csv for every job/family, including n=0 and n<50. Recall is missing when no attack row from that family occurs; it is not zero. All small families are retained. The 50-row flag is a descriptive aid only.\nThe direction/model table averages supported seed jobs equally (up to five). The target/model table averages supported source/seed jobs equally (up to fifteen). These are descriptive job averages, not pooled independent-flow estimates or confidence intervals. Do not interpret repeated rows across jobs/seeds as unique flows.\nFamily names are exact target dataset labels. This is binary attack detection within each family, not multiclass classification, a family-held-out experiment or evidence that a source-only warning predicts family-specific failure. Balanced original panels remain balanced. No new operational prevalence evaluation is performed.\nOriginal models, predictions and thresholds are unchanged. Reproduction checks compare counts and recalls to all 180 original target jobs. Recovered label alignment does not replay all 49 raw feature values.\n\'\'\')\n        write_json(output/\'progress.json\',{\'status\':\'PASS\',\'completed_jobs\':len(checks)})\n    except Exception as e:\n        write_json(output/\'verification_summary.json\',{\'status\':\'FAILED\',\'error\':str(e),\'completed_jobs\':len(locals().get(\'checks\',[]))})\n        write_json(output/\'progress.json\',{\'status\':\'FAILED\',\'error\':str(e)})\n        print(\'FAILED:\',str(e),flush=True)\n    hashes={p.name:sha(p) for p in sorted(output.iterdir()) if p.is_file()}\n    write_json(output/\'output_hashes.json\',hashes)\n    package=output.with_suffix(\'.zip\')\n    with zipfile.ZipFile(package,\'x\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(output.iterdir()):\n            if p.is_file():z.write(p,output.name+\'/\'+p.name)\n    print(\'Result status:\',read_json(output/\'verification_summary.json\')[\'status\']);print(\'Results ZIP:\',package)\n    return package\n'
exec(compile(EVALUATION_SOURCE, "paper3_attack_family_evaluation.py", "exec"))

In [ ]:
stamp = datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
output = PAPER3_ROOT / 'supplementary_attack_family_evaluation_20261024' / stamp
results_zip = evaluate(PAPER3_ROOT, RECOVERY_ROOT, output)
from google.colab import files
files.download(str(results_zip))

## Send the results back

Attach the downloaded **run_….zip** here. If the status is `FAILED`, send that ZIP too so we can inspect the exact failure. Do not edit the original files or weaken a verification check.

If the download does not start, find the newest ZIP in **Paper3_DataSet → supplementary_attack_family_evaluation_20261024**. The corresponding folder contains the plan, per-job family table, direction/model and target/model summaries, family support counts, original-binary-result checks and verification summary.

No model performance conclusion has been inserted into the manuscript yet. We will inspect every result and small-family count before deciding what to report.